<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/src/ieee_transfer_and_drift_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real-data transfer (IEEE-CIS) and month-by-month drift (Sparkov)

**Author:** Trilok Kumar Reddy (Data & EDA lead) · COMP8851 Group 2 · supervisor Dr. Mahmoud Bekhit

Two questions that reviewers will ask about our paper:

1. **Does the temporal signal transfer to real data?** Sparkov is synthetic. IEEE-CIS is real, but it has **no card number and no geography**. So we (a) build a card identifier from `card1 + addr1 + account start date` (start date = transaction day minus `D1`), (b) build time-gap and burst features per card, and (c) test whether they add predictive power on real transactions. We also run the *same* temporal features on Sparkov so the two datasets can be compared like-for-like.
2. **Does performance drift over time?** We score the team's final Sparkov model separately for each test month and report how the fraud rate, the input features and the metrics change. The final model is the Random Forest Tuned with the **time-gap feature in place of implied velocity** (whole-test PR-AUC 0.8444).

**Protocol (same as `time_honest_evaluation.ipynb`):** time-based train / validation / test split, F2-optimal threshold chosen on validation only, several seeds, and a 95% confidence interval from a **day-block bootstrap** (`src/evaluation.py`). Differences between feature sets are paired differences on the same resampled days. Nothing in this notebook uses the test set to choose anything.

**Where this notebook differs from the team's evaluation notebook (so the numbers are not compared blindly):**

| | Team's `time_honest_evaluation.ipynb` | This notebook |
|---|---|---|
| Bootstrap resamples | 500 | 200 (the intervals are slightly noisier at their ends) |
| Drift model (Section 3) | 200 trees, 5 seeds | the same: 200 trees, 5 seeds |
| Feature-set comparisons (Sections 4 and 5) | not applicable | 3 seeds and 100 trees per Random Forest, 400 trees for LightGBM |
| `max_features` | fixed at 3 | chosen on validation for each feature set (1 seed, 60 trees) |
| Training rows | all | the first 30 days are skipped as training rows (card-history burn-in) |

**Run order:** run all cells top to bottom. The Colab-only cells (marked `# [COLAB-ONLY]`) download the data and the repo. Total time is roughly 1.5–2.5 hours on a free Colab CPU; progress is saved to Google Drive so a disconnect only loses the seed that was running.

**Before running:** (1) you must have joined the IEEE-CIS competition on Kaggle (rules accepted, phone verified); (2) have your Kaggle key and a GitHub token ready. They are typed in at runtime with `getpass` and are never saved in the notebook.

## 1. Setup

In [ ]:
# [COLAB-ONLY] 1.1 Download the data from Kaggle (credentials are typed in, never stored)
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')

!kaggle datasets download -d kartik2112/fraud-detection -p /content/data/sparkov --unzip
!kaggle competitions download -c ieee-fraud-detection -f train_transaction.csv -p /content/data/ieee
!unzip -o -q /content/data/ieee/train_transaction.csv.zip -d /content/data/ieee

In [ ]:
# [COLAB-ONLY] 1.1b IEEE-CIS fallback. If the Kaggle download above printed "401 Unauthorized", your Kaggle account has not
# accepted the competition rules / finished phone verification for the API. Use the zip you downloaded from the Kaggle website instead:
# run this cell, click "Choose files", and select ieee-fraud-detection.zip (or train_transaction.csv).
import os, zipfile
os.makedirs('/content/data/ieee', exist_ok=True)
target = '/content/data/ieee/train_transaction.csv'
if not os.path.exists(target):
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith('.zip'):
            with zipfile.ZipFile(name) as z:
                z.extract('train_transaction.csv', '/content/data/ieee')
        elif name.endswith('train_transaction.csv'):
            os.replace(name, target)
print('train_transaction.csv:', round(os.path.getsize(target) / 1e6), 'MB (expect about 683 MB)')

Saving ieee-fraud-detection.zip to ieee-fraud-detection.zip
train_transaction.csv: 683 MB (expect about 683 MB)


In [ ]:
# [COLAB-ONLY] 1.2 Clone the project repository (contains src/split.py, src/utils.py, src/evaluation.py)
github_token = getpass('Paste your GitHub token: ')
repo_url = f'https://{github_token}@github.com/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-.git'
!git clone {repo_url} 2>&1 | sed 's#https://[^@]*@#https://#'

In [ ]:
# [COLAB-ONLY] 1.2b Repo fallback. If the clone above failed (fine-grained GitHub tokens cannot read a repo owned by someone else),
# run this cell and upload the repo ZIP ("Code -> Download ZIP" on GitHub). If the repo cloned fine, this cell does nothing.
import os, glob, shutil, zipfile
REPO_DIR = '/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'
if not os.path.isdir(REPO_DIR + '/src'):
    from google.colab import files
    uploaded = files.upload()
    shutil.rmtree('/content/_repo_zip', ignore_errors=True)
    shutil.rmtree(REPO_DIR, ignore_errors=True)      # remove any empty folder left by a failed clone
    for name in uploaded:
        if name.endswith('.zip'):
            zipfile.ZipFile(name).extractall('/content/_repo_zip')
    shutil.move(glob.glob('/content/_repo_zip/*')[0], REPO_DIR)
print(sorted(os.listdir(REPO_DIR)))

Saving Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning--main.zip to Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning--main.zip
['.gitignore', 'README.md', 'data', 'notebooks', 'requirements.txt', 'results', 'src']


In [ ]:
# [COLAB-ONLY] 1.3 Move into the repository and place the data in data/
%cd /content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-
!mkdir -p data && mv /content/data/sparkov /content/data/ieee data/

/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


In [ ]:
# [COLAB-ONLY] 1.4 Connect Google Drive so finished models survive a disconnect
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 1.5 Imports, settings and project helpers
import os, sys, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

sys.path.append(os.getcwd())
from src.split import split_sparkov
from src.utils import fit_category_risk, apply_category_risk, add_prev_txn_features
from src.evaluation import (chronological_val_split, run_seeds, bootstrap_model,
                            summarise, paired_difference, format_ci, point_metrics)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# FAST_MODE = a quick smoke test (tiny models, 2 seeds). NEVER use its numbers in the paper.
FAST_MODE = os.environ.get('FAST_MODE', '0') == '1'

SEEDS        = [42, 43, 44, 45, 46]   # drift model: same seeds as time_honest_evaluation.ipynb
ABL_SEEDS    = [42, 43, 44]           # feature-set comparisons (many more fits, so 3 seeds)
N_TREES_DRIFT = 200                   # Random Forest Tuned = 200 trees, max_features=3, min_samples_leaf=5
N_TREES_ABL   = 100
N_BOOT        = 200                   # day-block bootstrap resamples
BURN_IN_DAYS  = 30                    # first days of the data are not used as TRAINING rows (card history is cut off there)
N_TREES_SEL   = 60                    # trees used only to choose max_features on the validation part
N_LGB         = 400                   # LightGBM trees
MF_GRID_IEEE  = [3, 'sqrt', 0.3, 0.6] # max_features values tried (3 = the Sparkov setting)
MF_GRID_SP    = [3, 'sqrt', 0.5]
if FAST_MODE:
    SEEDS, ABL_SEEDS, N_TREES_DRIFT, N_TREES_ABL, N_BOOT = [42, 43], [42, 43], 20, 20, 20
    N_TREES_SEL, N_LGB = 10, 30

# where finished seeds are saved (re-running skips anything already saved)
DRIVE = '/content/drive/MyDrive/fraud_checkpoints'
use_drive = os.path.isdir('/content/drive/MyDrive') and not FAST_MODE
team_dir  = f'{DRIVE}/time_honest_evaluation'          # Subba's folder: lets us re-use his saved RF seeds
DRIFT_CKPT = team_dir if (use_drive and os.path.isdir(team_dir)) else ('ckpt_fast/drift' if FAST_MODE else (f'{DRIVE}/ieee_drift' if use_drive else 'ckpt/drift'))
ABL_CKPT   = 'ckpt_fast/ablation' if FAST_MODE else (f'{DRIVE}/ieee_transfer' if use_drive else 'ckpt/ablation')
os.makedirs('results', exist_ok=True)
print('FAST_MODE:', FAST_MODE, '| drift checkpoints:', DRIFT_CKPT, '| ablation checkpoints:', ABL_CKPT)

FAST_MODE: False | drift checkpoints: /content/drive/MyDrive/fraud_checkpoints/ieee_drift | ablation checkpoints: /content/drive/MyDrive/fraud_checkpoints/ieee_transfer


### 1.6 The card identifier and the temporal features (tested helpers)

These functions are used on **both** datasets.

* **Card identifier (IEEE-CIS).** `D1` is "days since this card's first transaction". Subtracting it from the transaction's day number gives the day the account started, which is the same for every transaction of that account. Combined with `card1` and `addr1`, it identifies one account. It is an **approximation** (two different cards could share all three values, and a missing `addr1` or `D1` gets its own code), and the paper should say so.
* **Temporal features.** All are **backward-looking**: a row only sees earlier rows of the same card, never later rows and never a fraud label, so they are safe to compute before splitting (the same reasoning as `implied_velocity` in `src/utils.py`). Features: time since the previous transaction (`gap_log`, with an `is_first` flag), number of earlier transactions (`prior_count_log`), bursts = number of earlier transactions in the last 10 min / 1 h / 24 h, and how unusual the amount is against the card's own history (`amt_vs_hist`). Counts are log1p-transformed.
* The cell after the helpers **checks them**: a hand-made example with known answers, and a test that changing future rows leaves earlier rows' features unchanged.

We define the card ID and the time features. Each feature only looks at that card's earlier transactions, never the future or the fraud label.

In [ ]:
import numpy as np
import pandas as pd

GAP_FILL_SECONDS = 30 * 86400   # "no earlier transaction" is treated as a 30-day gap
BURST_WINDOWS = {'burst_10m': 600, 'burst_1h': 3600, 'burst_24h': 86400}


def make_card_uid(card1, addr1, d1, t_seconds):
    """Card identifier = card1 + addr1 + account start day (from D1).

    account start day = (day number of the transaction) - D1, so every transaction of the
    same account lands on the same start day. Missing addr1 / D1 are given their own code,
    so a missing value never merges two different cards by accident beyond that card's group.
    Returns integer codes (one per distinct identifier).
    """
    day = np.floor(np.asarray(t_seconds, dtype=float) / 86400.0)
    start_day = day - np.asarray(d1, dtype=float)                    # NaN where D1 is missing
    start_day = np.where(np.isnan(start_day), -9999, start_day).astype(np.int64)
    addr = np.where(pd.isna(addr1), -1, addr1).astype(np.int64)
    key = (pd.Series(np.asarray(card1)).astype(str) + '_' + pd.Series(addr).astype(str)
           + '_' + pd.Series(start_day).astype(str))
    return pd.factorize(key)[0]


def temporal_features(uid, t_seconds, amount):
    """Backward-looking features per card identifier. Row i only uses rows with an earlier
    (or equal-time, earlier-listed) timestamp in the same identifier, never later rows and
    never the fraud label.

      gap_log          log1p(seconds since this card's previous transaction)
      is_first         1 if the card has no earlier transaction
      prior_count_log  log1p(number of earlier transactions on this card)
      burst_10m/1h/24h log1p(number of earlier transactions within the last 10 min / 1 h / 24 h)
      amt_vs_hist      log1p(amount) - log1p(mean amount of this card's earlier transactions)
    """
    uid = np.asarray(uid, dtype=np.int64)
    t = np.asarray(t_seconds, dtype=np.int64)
    a = np.asarray(amount, dtype=float)
    n = len(t)

    order = np.lexsort((np.arange(n), t, uid))          # by card, then time, then original order
    u, ts, am = uid[order], t[order] - t.min(), a[order]
    idx = np.arange(n)

    new_group = np.r_[True, u[1:] != u[:-1]]
    group_start = np.maximum.accumulate(np.where(new_group, idx, 0))
    prior_count = idx - group_start                      # earlier transactions on this card

    gap = np.full(n, np.nan)
    gap[1:] = ts[1:] - ts[:-1]
    gap[new_group] = np.nan
    is_first = new_group.astype(float)
    gap_log = np.log1p(np.where(np.isnan(gap), GAP_FILL_SECONDS, np.maximum(gap, 0)))

    big = int(ts.max()) + 86400 + 2
    key = u * big + ts
    out = {'gap_log': gap_log, 'is_first': is_first,
           'prior_count_log': np.log1p(prior_count)}
    for name, w in BURST_WINDOWS.items():
        left = np.searchsorted(key, u * big + np.maximum(ts - w, 0), side='left')
        out[name] = np.log1p(idx - left)                 # rows in this card's [t-w, now) window

    cs = np.cumsum(am)
    before_group = cs[group_start] - am[group_start]
    prior_sum = (cs - am) - before_group
    with np.errstate(divide='ignore', invalid='ignore'):
        prior_mean = np.where(prior_count > 0, prior_sum / np.maximum(prior_count, 1), np.nan)
    out['amt_vs_hist'] = np.where(np.isnan(prior_mean), 0.0, np.log1p(am) - np.log1p(prior_mean))

    result = pd.DataFrame(index=np.arange(n), columns=list(out), dtype=float)
    for k, v in out.items():
        col = np.empty(n)
        col[order] = v                                   # back to the original row order
        result[k] = col
    return result

In [ ]:
# 1.7 Check the helpers (these asserts must pass before anything else is trusted)
uid_ = [0, 0, 0, 1, 0]; t_ = [0, 100, 700, 50, 800]; a_ = [10, 20, 30, 5, 40]
f_ = temporal_features(uid_, t_, a_)
assert f_.is_first.tolist() == [1, 0, 0, 1, 0]
assert np.allclose(np.expm1(f_.gap_log)[[1, 2, 4]], [100, 600, 100])          # gaps in seconds
assert np.allclose(np.expm1(f_.burst_10m), [0, 1, 1, 0, 1])                   # earlier txns within 600 s
assert np.allclose(np.expm1(f_.burst_1h), [0, 1, 2, 0, 3])
assert np.allclose(np.expm1(f_.prior_count_log), [0, 1, 2, 0, 3])
assert np.isclose(f_.amt_vs_hist[2], np.log1p(30) - np.log1p(15))              # 30 vs mean(10, 20)

rng_ = np.random.default_rng(0); n_ = 5000
uid_ = rng_.integers(0, 300, n_); t_ = np.sort(rng_.integers(0, 10**7, n_)); a_ = rng_.gamma(2, 50, n_)
f1_ = temporal_features(uid_, t_, a_)
t2_, a2_ = t_.copy(), a_.copy(); t2_[3000:] += 12345; a2_[3000:] *= 7           # change only the FUTURE rows
f2_ = temporal_features(uid_, t2_, a2_)
assert np.allclose(f1_.iloc[:3000].values, f2_.iloc[:3000].values)             # earlier rows must not change
print('helper checks passed: values match the hand calculation and no feature looks at later rows')

helper checks passed: values match the hand calculation and no feature looks at later rows


Tests the timing features. All checks passed ("helper checks passed"), so the features match a hand calculation and never look at later rows.

## 2. Sparkov data: the paper's features, plus the temporal features

Rebuilt exactly as in `time_honest_evaluation.ipynb` (velocity on the combined train + test timeline, `category_risk` fitted later on the training part only). We additionally add the **same temporal features** (gap / bursts / amount-vs-history, per `cc_num`), so Section 5 can compare Sparkov with IEEE-CIS like-for-like and Section 3 can use the time-gap feature (`gap_log`) of the team's final model. Cell 2.2 lists the original six features with `velocity_log`; Section 3 swaps `velocity_log` for `gap_log`.

In [ ]:
# 2.1 Kaggle's chronological train/test split, implied velocity on the full timeline
TIME = 'trans_date_trans_time'
sparkov_train, sparkov_test = split_sparkov()
sparkov_train['_split'] = 'train'; sparkov_test['_split'] = 'test'
full = pd.concat([sparkov_train, sparkov_test]).sort_values(TIME)
full = add_prev_txn_features(full, 'cc_num', TIME, 'merch_lat', 'merch_long')
sparkov_train = full[full['_split'] == 'train'].drop(columns='_split')
sparkov_test  = full[full['_split'] == 'test'].drop(columns='_split')
del full

def add_features(df):
    df = df.copy()
    ts = pd.to_datetime(df[TIME])
    df['velocity_log'] = np.log1p(df['implied_velocity_kmh'].fillna(0))
    df['is_night']     = ts.dt.hour.isin([22, 23, 0, 1, 2, 3]).astype(int)
    df['amt_log']      = np.log1p(df['amt'])
    df['age']          = (ts - pd.to_datetime(df['dob'])).dt.days / 365.25
    df['gender_flag']  = (df['gender'] == 'M').astype(int)
    df['day']          = ts.dt.floor('D')          # used only by the bootstrap
    return df
sparkov_train = add_features(sparkov_train)
sparkov_test  = add_features(sparkov_test)

# temporal features per card (cc_num) on the combined timeline, then split back
n_tr = len(sparkov_train)
uid_sp = pd.factorize(np.concatenate([sparkov_train['cc_num'].values, sparkov_test['cc_num'].values]))[0]
t_sp   = np.concatenate([sparkov_train['unix_time'].values, sparkov_test['unix_time'].values])
a_sp   = np.concatenate([sparkov_train['amt'].values, sparkov_test['amt'].values])
tf_sp  = temporal_features(uid_sp, t_sp, a_sp)
for c in tf_sp.columns:
    sparkov_train[c] = tf_sp[c].values[:n_tr]
    sparkov_test[c]  = tf_sp[c].values[n_tr:]
del tf_sp
print(sparkov_train.shape, sparkov_test.shape)

Train: 1,296,675 rows, from 2019-01-01 00:00:18 to 2020-06-21 12:13:37
Test:  555,719 rows, from 2020-06-21 12:14:25 to 2020-12-31 23:59:34
(1296675, 39) (555719, 39)


Loads Sparkov in time order. We got 1,296,675 training rows and 555,719 test rows.

In [ ]:
# 2.2 Time-based validation split; category_risk fitted on the training part only
train_sub, val, cutoff = chronological_val_split(sparkov_train, TIME, val_frac=0.2)
category_risk_map = fit_category_risk(train_sub, 'category', 'is_fraud')
for df in (train_sub, val, sparkov_test):
    df['category_risk'] = apply_category_risk(df, 'category', category_risk_map)

FEATURES = ['amt_log', 'category_risk', 'is_night', 'velocity_log', 'age', 'gender_flag']
X_tr,  y_tr  = train_sub[FEATURES],    train_sub['is_fraud']
X_val, y_val = val[FEATURES],          val['is_fraud']
X_test, y_test = sparkov_test[FEATURES], sparkov_test['is_fraud']
print('cutoff:', cutoff)
for name, df in [('train_sub', train_sub), ('val', val), ('test', sparkov_test)]:
    t = pd.to_datetime(df[TIME])
    print(f"{name:9s} rows={len(df):>9,}  fraud rate={df['is_fraud'].mean():.4%}  {t.min():%Y-%m-%d} -> {t.max():%Y-%m-%d}")

cutoff: 2020-03-06 00:10:57.200000
train_sub rows=1,036,901  fraud rate=0.5756%  2019-01-01 -> 2020-03-06
val       rows=  259,774  fraud rate=0.5921%  2020-03-06 -> 2020-06-21
test      rows=  555,719  fraud rate=0.3860%  2020-06-21 -> 2020-12-31


Holds out the last part of the training period for validation. We got a cutoff of 2020-03-06, with 1,036,901 / 259,774 / 555,719 rows for train, validation and test.

## 3. Month-by-month performance on Sparkov (drift)

We use the paper's final model: the **Random Forest Tuned with the time-gap feature (`gap_log`) in place of implied velocity** (200 trees, `max_features=3`, `min_samples_leaf=5`; whole-test PR-AUC 0.8444 in the team's evaluation). It is trained once on the training part with its threshold chosen once on validation. We then **do not retrain**: we only slice the test period by calendar month and score each slice. If the team's saved seeds are in your Drive (`velocity_breakdown` folder), they are loaded; otherwise the five seeds are trained (cell 3.1 prints which files it found, and the difference from 0.8444).

For every month we report: the **fraud rate**, **PR-AUC with a 95% day-block bootstrap interval**, **ROC-AUC** (it does not depend on how rare fraud is, so it shows whether the model's *ranking* got worse), **lift** (PR-AUC divided by that month's fraud rate; shown for reference, but it rises mechanically when fraud gets rarer, so do not read it as quality), and F2 / MCC / precision / recall at the fixed validation threshold. The first test month (2020-06) only contains 21–30 June.

In [ ]:
# 3.1 The final model: Random Forest Tuned with the TIME-GAP feature in place of implied velocity
FEATURES = ['amt_log', 'category_risk', 'is_night', 'gap_log', 'age', 'gender_flag']   # velocity_log -> gap_log
X_tr,  y_tr  = train_sub[FEATURES],     train_sub['is_fraud']
X_val, y_val = val[FEATURES],           val['is_fraud']
X_test, y_test = sparkov_test[FEATURES], sparkov_test['is_fraud']

# Saved seeds: the team's `velocity_breakdown` folder if it is in your Drive, otherwise a folder of our own.
GAP_TEAM_DIR = f'{DRIVE}/velocity_breakdown'
GAP_NAME = 'Random Forest Gap'     # file prefix of the saved seeds; edit it if the team's files use another prefix
GAP_CKPT = GAP_TEAM_DIR if (use_drive and os.path.isdir(GAP_TEAM_DIR)) else (
    'ckpt_fast/drift' if FAST_MODE else (f'{DRIVE}/ieee_drift' if use_drive else 'ckpt/drift'))
if os.path.isdir(GAP_CKPT):
    print('saved seed files in', GAP_CKPT, ':', sorted(f for f in os.listdir(GAP_CKPT) if f.endswith('seed42.npz')))

make_rf = lambda s: RandomForestClassifier(n_estimators=N_TREES_DRIFT, max_features=3, min_samples_leaf=5,
                                           random_state=s, n_jobs=-1)
rf_runs = run_seeds(GAP_NAME, make_rf, X_tr, y_tr, X_val, y_val, X_test, SEEDS, checkpoint_dir=GAP_CKPT)
assert all(len(r['test_proba']) == len(X_test) for r in rf_runs), 'saved predictions do not match this test set - use another checkpoint folder'

pairs = [(r['test_proba'], r['threshold']) for r in rf_runs]
overall = pd.DataFrame([point_metrics(y_test.values, p, t) for p, t in pairs]).mean()
print('\nWhole test period (the team reports PR-AUC 0.8444 for the time-gap model):')
print(overall.round(4).to_string())
print(f"Difference from the team's value: {overall['PR-AUC'] - 0.8444:+.4f}")

Loads or trains the final model (the time-gap Random Forest) and prints its whole-test PR-AUC next to the team's 0.8444. The two should agree to about the third decimal.

In [ ]:
# 3.2 Score each test month separately
test_ts   = pd.to_datetime(sparkov_test[TIME])
months    = test_ts.dt.to_period('M').astype(str).values
days_arr  = sparkov_test['day'].values
y_arr     = y_test.values

from sklearn.metrics import roc_auc_score

def monthly_report(y, runs, months, days, n_boot, boot_seed=2026):
    rows = []
    for m in sorted(set(months)):
        mk = months == m
        ym = y[mk]
        if ym.sum() < 5:
            print(f'{m}: only {int(ym.sum())} frauds - skipped'); continue
        mpairs = [(r['test_proba'][mk], r['threshold']) for r in runs]
        boot = bootstrap_model(ym, mpairs, days[mk], n_boot=n_boot, boot_seed=boot_seed)
        s = summarise(ym, mpairs, boot)
        n_days = len(np.unique(days[mk]))
        rate = ym.mean()
        pm = pd.DataFrame([point_metrics(ym, p, t) for p, t in mpairs]).mean()
        roc = np.mean([roc_auc_score(ym, p) for p, t in mpairs])      # does not depend on the fraud rate
        rows.append({'Month': m + (' (partial)' if n_days < 25 else ''), 'Days': n_days,
                     'Transactions': int(mk.sum()), 'Frauds': int(ym.sum()), 'Fraud rate %': 100 * rate,
                     'PR-AUC': s['PR-AUC'], 'PR-AUC low': s['PR-AUC CI low'], 'PR-AUC high': s['PR-AUC CI high'],
                     'Lift (PR-AUC / fraud rate)': s['PR-AUC'] / rate, 'ROC-AUC': roc,
                     'F2': s['F2'], 'F2 low': s['F2 CI low'], 'F2 high': s['F2 CI high'],
                     'MCC': s['MCC'], 'Precision': pm['Precision'], 'Recall': pm['Recall'],
                     'Alert rate %': 100 * np.mean([(p >= t).mean() for p, t in mpairs]),
                     'Mean predicted prob %': 100 * np.mean([p.mean() for p, t in mpairs])})
    return pd.DataFrame(rows)

monthly = monthly_report(y_arr, rf_runs, months, days_arr, N_BOOT)
monthly.to_csv('results/sparkov_monthly_drift.csv', index=False)
show = monthly.copy()
show['PR-AUC [95% CI]'] = [f"{r['PR-AUC']:.3f} [{r['PR-AUC low']:.3f}, {r['PR-AUC high']:.3f}]" for _, r in monthly.iterrows()]
show['F2 [95% CI]'] = [f"{r['F2']:.3f} [{r['F2 low']:.3f}, {r['F2 high']:.3f}]" for _, r in monthly.iterrows()]
show[['Month', 'Days', 'Transactions', 'Frauds', 'Fraud rate %', 'PR-AUC [95% CI]', 'Lift (PR-AUC / fraud rate)', 'ROC-AUC',
      'F2 [95% CI]', 'MCC', 'Precision', 'Recall', 'Alert rate %', 'Mean predicted prob %']].round(3)

Scores each month separately. PR-AUC is about 0.84 to 0.89 from June to November and falls to about 0.79 in December; ROC-AUC stays between about 0.98 and 0.99 in every month.

### 3.3 Did the inputs drift? Population Stability Index (PSI) per feature and month

PSI compares a month's feature distribution with the **training part** (reference). Common rule of thumb: below 0.1 = stable, 0.1–0.25 = moderate shift, above 0.25 = major shift. It is a diagnostic of *input* drift; it does not use fraud labels.

In [ ]:
def psi(ref, cur, bins=10):
    ref = np.asarray(ref, float); cur = np.asarray(cur, float)
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:                                   # binary or near-constant feature
        v = np.unique(ref); edges = np.r_[v.min() - .5, (v[:-1] + v[1:]) / 2, v.max() + .5]
    edges = edges.astype(float); edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref); c = np.histogram(cur, edges)[0] / len(cur)
    r, c = np.clip(r, 1e-4, None), np.clip(c, 1e-4, None)
    return float(np.sum((c - r) * np.log(c / r)))

month_labels = list(monthly['Month'])
psi_rows = {}
for lab in month_labels:
    m = lab.split(' ')[0]
    sel = months == m
    psi_rows[lab] = {f: psi(X_tr[f], X_test.loc[sel, f]) for f in FEATURES}
psi_df = pd.DataFrame(psi_rows).T
psi_df.to_csv('results/sparkov_monthly_psi.csv')
psi_df.round(3)

PSI compares each month's inputs with training. Every feature stays at or below about 0.03 in every month, except `gap_log` in December (about 0.21, moderate). This is the shift we expected: December has about twice the transactions, so cards have more transactions each and shorter gaps between them. The old velocity feature showed the same pattern (0.155 in December).

In [ ]:
# 3.4 Drift figure
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
x = np.arange(len(monthly))
lab = [m.replace(' (partial)', '*') for m in monthly['Month']]

a = ax[0, 0]
a.errorbar(x, monthly['PR-AUC'], yerr=[monthly['PR-AUC'] - monthly['PR-AUC low'], monthly['PR-AUC high'] - monthly['PR-AUC']],
           fmt='o-', capsize=4, color='#1f5fa8')
a.axhline(overall['PR-AUC'], ls='--', color='grey', lw=1, label='whole test period')
a.set_xticks(x); a.set_xticklabels(lab, rotation=30); a.set_ylabel('PR-AUC'); a.set_title('PR-AUC by month (95% day-block CI)'); a.legend()

a = ax[0, 1]
a.bar(x, monthly['Fraud rate %'], color='#d98324'); a.set_xticks(x); a.set_xticklabels(lab, rotation=30)
a.set_ylabel('fraud rate (%)'); a.set_title('Monthly fraud rate')
a2 = a.twinx(); a2.plot(x, monthly['Lift (PR-AUC / fraud rate)'], 'k.-'); a2.set_ylabel('lift (PR-AUC / fraud rate)')

a = ax[1, 0]
a.plot(x, monthly['Recall'], 'o-', label='recall'); a.plot(x, monthly['Precision'], 's-', label='precision')
a.plot(x, monthly['F2'], '^-', label='F2'); a.set_xticks(x); a.set_xticklabels(lab, rotation=30)
a.set_ylim(0, 1); a.set_title('Fixed validation threshold, scored per month'); a.legend()

a = ax[1, 1]
im = a.imshow(psi_df.T.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=0.3)
a.set_xticks(range(len(psi_df))); a.set_xticklabels(lab, rotation=30); a.set_yticks(range(len(FEATURES))); a.set_yticklabels(FEATURES)
for i in range(len(FEATURES)):
    for j in range(len(psi_df)):
        a.text(j, i, f'{psi_df.T.values[i, j]:.2f}', ha='center', va='center', fontsize=8)
a.set_title('Feature PSI vs training (0.1 moderate, 0.25 major)'); plt.colorbar(im, ax=a)
plt.suptitle('Sparkov drift: Random Forest Tuned with the time-gap feature, no retraining   (* = partial month)', y=1.01)
plt.tight_layout(); plt.savefig('results/sparkov_monthly_drift.png', dpi=150, bbox_inches='tight'); plt.show()

full_mo = monthly[~monthly['Month'].str.contains('partial')]
first, last = full_mo.iloc[0], full_mo.iloc[-1]
overlap = not (first['PR-AUC high'] < last['PR-AUC low'] or last['PR-AUC high'] < first['PR-AUC low'])
print(f"PR-AUC range across months: {monthly['PR-AUC'].min():.3f} to {monthly['PR-AUC'].max():.3f}; "
      f"fraud rate range: {monthly['Fraud rate %'].min():.3f}% to {monthly['Fraud rate %'].max():.3f}%")
print(f"First vs last full month PR-AUC ({first['Month']} vs {last['Month']}): "
      f"{first['PR-AUC']:.3f} vs {last['PR-AUC']:.3f}; 95% intervals {'overlap' if overlap else 'do NOT overlap'}")
print(f"Largest feature PSI in any month: {psi_df.values.max():.3f} ({psi_df.stack().idxmax()})")

The four panels: PR-AUC by month, fraud rate by month, recall / precision / F2 at the fixed threshold, and the PSI heat map. December has the shortest fraud-rate bar, the lowest PR-AUC, and the one moderate PSI cell (`gap_log`).

In [ ]:
# 3.5 Why precision fell in December: false-alarm rate per month (computed from the table above)
chk = monthly.copy()
tp = chk['Recall'] * chk['Frauds']
fp = tp * (1 / chk['Precision'] - 1)
chk['False-alarm rate % (of genuine txns)'] = 100 * fp / (chk['Transactions'] - chk['Frauds'])
chk[['Month', 'Transactions', 'Fraud rate %', 'Recall', 'Precision', 'ROC-AUC', 'False-alarm rate % (of genuine txns)']].round(3)

Why December is lower: the false-alarm rate (share of genuine transactions that raise an alarm) stays at about 0.18% to 0.21% in every month, December included, and recall stays near 0.87. The model flags genuine transactions at the same rate; there are simply fewer frauds to find, so precision falls.

### 3.6 Findings: how does the model behave over time on Sparkov?

*(Model: Random Forest Tuned with the time-gap feature, the team's final model. Values are from tables 3.2, 3.3 and 3.5, rounded; read the exact figures there.)*

1. **Ranking quality is stable.** ROC-AUC stays between about 0.98 and 0.99 in every month, and PR-AUC is about 0.84 to 0.89 from June to November.
2. **December is the only weaker month**, with PR-AUC about 0.79. The 95% intervals of neighbouring months overlap, so the drop is small compared with the month-to-month noise, but it is the lowest value in the table.
3. **The December drop looks mostly like a base-rate effect, not worse ranking.** Fraud became rarer (0.185% against 0.37% to 0.55% in the other months) while the transaction volume roughly doubled (139,538 against 69,000 to 89,000). Recall stayed near 0.87 and ROC-AUC near 0.99, and the false-alarm rate stayed at about 0.18% to 0.21% (table 3.5). With the same false-alarm rate spread over fewer frauds, precision falls from about 0.63 to 0.72 to about 0.45, and PR-AUC falls with it.
4. **The inputs barely drifted, with one exception that is linked to the volume.** PSI is about 0.03 or lower for every feature in every month, except **`gap_log` in December (about 0.21, moderate)**. More transactions per card means shorter gaps, so a shift in `gap_log` is expected when the volume doubles. The time-gap feature is therefore sensitive to volume changes. We did **not** test whether this shift causes the weaker December numbers.
5. **The fixed threshold keeps its operating point.** Recall stays between about 0.82 and 0.88 in all months, so the validation threshold did not go stale; precision moves with the fraud rate.

**Limits of this result:** Sparkov is simulated by one generator, so stability is expected and is a statement about the simulator, which is why the real-data check in Section 4 matters. There is one test window (June to December 2020), no retraining, and the cause of the December change (the doubled volume, the lower fraud rate and the `gap_log` shift happen together) was not separated.

## 4. IEEE-CIS (real data): card identifier and temporal features

Only the columns we need are loaded (`train_transaction.csv`; the `test_*` files have no labels, so they cannot be scored). `TransactionDT` is seconds from a hidden reference date, so **hour-of-day and `is_night` are not reliable here and are not used**. The date `2017-11-30` used below is the commonly assumed origin and is only used to order and group days.

In [ ]:
# 4.1 Load IEEE-CIS and build the card identifier
IEEE_COLS = ['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD',
             'card1', 'card4', 'card6', 'addr1', 'D1'] + [f'C{i}' for i in range(1, 15)]
ieee = pd.read_csv('data/ieee/train_transaction.csv', usecols=IEEE_COLS).sort_values('TransactionDT').reset_index(drop=True)
ieee['ts']  = pd.Timestamp('2017-11-30') + pd.to_timedelta(ieee['TransactionDT'], unit='s')   # assumed origin
ieee['day'] = ieee['ts'].dt.floor('D')
ieee['uid'] = make_card_uid(ieee['card1'], ieee['addr1'], ieee['D1'], ieee['TransactionDT'])
print(f"{len(ieee):,} transactions, fraud rate {ieee['isFraud'].mean():.3%}, "
      f"{ieee['ts'].min():%Y-%m-%d} -> {ieee['ts'].max():%Y-%m-%d}")
print(f"missing: addr1 {ieee['addr1'].isna().mean():.1%}, D1 {ieee['D1'].isna().mean():.2%}")

590,540 transactions, fraud rate 3.499%, 2017-12-01 -> 2018-05-31
missing: addr1 11.1%, D1 0.21%


We loaded the real data and built a card ID. We got 590,540 transactions.

### 4.2 Is the identifier meaningful? (diagnostics)

If the identifier really groups transactions of one account, **fraud should cluster inside identifiers**. We compare, for the real identifier and for a **shuffled** one (same group sizes, random members), the share of fraud transactions that sit in an identifier with at least 2 frauds. This uses labels, so it is only a diagnostic, never a feature.

In [ ]:
def cluster_share(uid, y):
    g = pd.DataFrame({'u': uid, 'y': y}).groupby('u')['y'].agg(['size', 'sum'])
    return g.loc[g['sum'] >= 2, 'sum'].sum() / g['sum'].sum(), g

share_real, g_real = cluster_share(ieee['uid'].values, ieee['isFraud'].values)
rng = np.random.default_rng(1)
share_shuf, _ = cluster_share(rng.permutation(ieee['uid'].values), ieee['isFraud'].values)
print(f"distinct card identifiers: {len(g_real):,} (card1 alone has {ieee['card1'].nunique():,})")
print(f"identifiers with a single transaction: {(g_real['size'] == 1).mean():.1%}; "
      f"transactions in identifiers with 2+ transactions: {g_real.loc[g_real['size'] > 1, 'size'].sum() / len(ieee):.1%}")
print(f"share of fraud in identifiers holding 2+ frauds:  real = {share_real:.1%}   shuffled = {share_shuf:.1%}")

distinct card identifiers: 217,850 (card1 alone has 13,553)
identifiers with a single transaction: 57.5%; transactions in identifiers with 2+ transactions: 78.8%
share of fraud in identifiers holding 2+ frauds:  real = 78.0%   shuffled = 20.7%


We checked that the ID is meaningful: 78% of fraud sits inside IDs with 2 or more frauds, against 21% when the IDs are shuffled.

In [ ]:
# 4.3 Features (backward-looking, no labels), plus a PLACEBO version built on shuffled identifiers
tf = temporal_features(ieee['uid'].values, ieee['TransactionDT'].values, ieee['TransactionAmt'].values)
rng = np.random.default_rng(7)
tf_pl = temporal_features(rng.permutation(ieee['uid'].values), ieee['TransactionDT'].values,
                          ieee['TransactionAmt'].values).add_suffix('_pl')
ieee = pd.concat([ieee, tf, tf_pl], axis=1)
ieee['amt_log'] = np.log1p(ieee['TransactionAmt'])
ieee['acct_age_days'] = ieee['D1'].fillna(-1)

# time split: last 20% = test; of the first 80%, the last 20% (by time) = validation
n = len(ieee)
test_i = ieee.iloc[int(n * 0.8):]
tv = ieee.iloc[:int(n * 0.8)]
tr_i, val_i, cut_i = chronological_val_split(tv, 'ts', val_frac=0.2)

cats = []
for c in ['ProductCD', 'card4', 'card6']:                      # categories fixed from the training part only
    for v in sorted(tr_i[c].dropna().unique()):
        ieee[f'{c}={v}'] = (ieee[c] == v).astype(float); cats.append(f'{c}={v}')
tr_i, val_i, test_i = ieee.loc[tr_i.index], ieee.loc[val_i.index], ieee.loc[test_i.index]

# --- Card-history burn-in -------------------------------------------------------------------------
# The data starts on a fixed day. A card whose earlier transactions happened BEFORE that day looks "new"
# (is_first = 1, prior count 0) in the first weeks, so the timing features would mean something different in
# training than in validation/test. The features are still computed on ALL rows (so later rows see the full
# history); we only skip the first BURN_IN_DAYS days as TRAINING rows.
week_ = ((ieee['ts'] - ieee['ts'].min()).dt.days // 7).rename('week')
hist = ieee.groupby(week_).agg(transactions=('isFraud', 'size'), share_that_look_like_a_first_transaction=('is_first', 'mean'))
print('Share of transactions that look like the first one of their card, by week since the data starts:')
display(hist.loc[[w for w in [0, 1, 2, 3, 4, 6, 8, 12, 16, 20, 25] if w in hist.index]].round(3))
hist.to_csv('results/ieee_card_history_by_week.csv')
burn_cut = ieee['ts'].min() + pd.Timedelta(days=BURN_IN_DAYS)
n_before, first_before = len(tr_i), tr_i['is_first'].mean()
tr_i = tr_i[tr_i['ts'] >= burn_cut]
print(f"Training rows: {n_before:,} -> {len(tr_i):,} after skipping the first {BURN_IN_DAYS} days. "
      f"Share of first-seen rows: train {first_before:.3f} -> {tr_i['is_first'].mean():.3f} "
      f"(validation {val_i['is_first'].mean():.3f}, test {test_i['is_first'].mean():.3f})")
for nm, d_ in [('train', tr_i), ('val', val_i), ('test', test_i)]:
    print(f"{nm:5s} rows={len(d_):>8,}  fraud rate={d_['isFraud'].mean():.3%}  {d_['ts'].min():%Y-%m-%d} -> {d_['ts'].max():%Y-%m-%d}")

# descriptive: fraud rate by temporal-feature bucket (TRAINING part only)
g_ = np.expm1(tr_i['gap_log']).where(tr_i['is_first'] == 0)
print('\nFraud rate by seconds since the card\'s previous transaction (training part):')
print(tr_i.groupby(pd.cut(g_, [-1, 60, 600, 3600, 86400, 7 * 86400, 1e12]))['isFraud'].agg(['mean', 'size']).round(4))
print('\nFraud rate by number of earlier transactions on the card in the last 24 h:')
print(tr_i.groupby(pd.cut(np.expm1(tr_i['burst_24h']), [-1, 0, 1, 3, 10, 1e9]))['isFraud'].agg(['mean', 'size']).round(4))

Share of transactions that look like the first one of their card, by week since the data starts:


,transactions,share_that_look_like_a_first_transaction
week,,
0,27596,0.660
1,28463,0.568
2,35701,0.483
3,34906,0.449
4,24539,0.367
6,20564,0.372
8,21432,0.347
12,22391,0.322
16,21078,0.310


Training rows: 389,881 -> 255,542 after skipping the first 30 days. Share of first-seen rows: train 0.404 -> 0.342 (validation 0.303, test 0.299)
train rows= 255,542  fraud rate=3.898%  2017-12-31 -> 2018-03-23
val   rows=  82,551  fraud rate=3.922%  2018-03-23 -> 2018-04-20
test  rows= 118,108  fraud rate=3.441%  2018-04-20 -> 2018-05-31

Fraud rate by seconds since the card's previous transaction (training part):
                               mean   size
gap_log                                   
(-1.0, 60.0]                 0.1418   3512
(60.0, 600.0]                0.0808  18614
(600.0, 3600.0]              0.0672  16270
(3600.0, 86400.0]            0.0606  33255
(86400.0, 604800.0]          0.0350  33282
(604800.0, 1000000000000.0]  0.0122  63334

Fraud rate by number of earlier transactions on the card in the last 24 h:
                        mean    size
burst_24h                           
(-1.0, 0.0]           0.0264  183891
(0.0, 1.0]            0.0557   34249
(1.0, 3.0]   

We checked whether the time features carry signal. Fraud is 14.2% when the card's last transaction was under a minute ago, against 1.2% after a week or more (or no earlier transaction). Cards also look new in week 0 (66%, against about 30% later), so we leave the first 30 days out of training only.

**Time features carry signal.** If a card's last transaction was under a minute ago, 14.2% of the transactions are fraud, against 1.2% when it was more than a week ago or the card had no earlier transaction. Fraud also rises with the number of recent transactions on the card, from 2.6% with none in the last 24 hours to 9.9% with 4 to 10.

**Cards look new at the start.** In week 0, 66% of transactions look like a card's first, falling to about 30% later, so we skipped the first 30 days as training rows (389,881 to 255,542 rows). The share of first-seen rows in training then becomes 0.342, close to validation (0.303) and test (0.299).

### 4.4 Does the temporal signal transfer? Feature-set comparison

Four models are compared on the same feature sets, because the answer can depend on the model:

* **LightGBM**: the standard model for IEEE-CIS (400 trees, learning rate 0.05, no class weights; the team's `boosting_comparison.ipynb` found that weighting hurts LightGBM).
* **Random Forest, `max_features` chosen on validation.** The paper's Random Forest uses `max_features=3`, which the team tuned for the 6 Sparkov features. The IEEE sets have 14 to 35 features, so 3 per split would unfairly penalise the bigger sets. For each feature set we therefore pick `max_features` from {3, sqrt, 0.3, 0.6} using the **validation part only** (never the test part).
* **Random Forest with `max_features=3`** (the Sparkov setting), kept as a sensitivity check so the effect of this setting is visible.
* **Logistic Regression** as a linear reference.

Feature sets:

| Set | Features |
|---|---|
| **A** | base: log amount, `ProductCD`, `card4`, `card6` |
| **B** | A + time-gap |
| **C** | A + burst counts (+ prior count) |
| **D** | A + time-gap + bursts |
| **E** | D + amount versus the card's own history |
| **F** | E + account age (`D1`) |
| **G** | A + IEEE's own count columns `C1`-`C14` (reference: "vendor-engineered" counts) |
| **H** | G + E's temporal features (do ours add anything **on top of** the vendor counts?) |
| **P** | **placebo**: E's features recomputed on *shuffled* identifiers. If P matches E, the gain was not from the card grouping. |

Every difference is a **paired** bootstrap difference on the same resampled test days. Training skips the first 30 days (card-history burn-in, see 4.3).

In [ ]:
# 4.5 Feature sets and the evaluation helpers
BASE = ['amt_log'] + cats
GAP, BUR, AH = ['gap_log', 'is_first'], ['prior_count_log', 'burst_10m', 'burst_1h', 'burst_24h'], ['amt_vs_hist']
CCOLS = [f'C{i}' for i in range(1, 15)]
IEEE_SETS = {
    'A': BASE, 'B': BASE + GAP, 'C': BASE + BUR, 'D': BASE + GAP + BUR, 'E': BASE + GAP + BUR + AH,
    'F': BASE + GAP + BUR + AH + ['acct_age_days'],
    'G': BASE + CCOLS, 'H': BASE + CCOLS + GAP + BUR + AH,
    'P': BASE + [c + '_pl' for c in GAP + BUR + AH],
}
LABEL = {'A': 'A base', 'B': 'B +gap', 'C': 'C +bursts', 'D': 'D +gap+bursts', 'E': 'E +amt vs history',
         'F': 'F +account age', 'G': 'G base+C1-C14', 'H': 'H G+temporal', 'P': 'P placebo of E'}

def eval_sets(tag, sets, tr, va, te, target, make_model, seeds, ckpt, n_boot, by_key=False):
    """by_key=True: make_model(key) returns the seed -> model function for that feature set."""
    runs, boots, summ = {}, {}, {}
    for key, cols in sets.items():
        mm = make_model(key) if by_key else make_model
        h = hashlib.md5((','.join(cols) + str(len(tr)) + str(sorted(mm(0).get_params().items()))).encode()).hexdigest()[:6]
        runs[key] = run_seeds(f'{tag}_{key}_{h}', mm, tr[cols], tr[target], va[cols], va[target],
                              te[cols], seeds, checkpoint_dir=ckpt)
        pr = [(r['test_proba'], r['threshold']) for r in runs[key]]
        boots[key] = bootstrap_model(te[target].values, pr, te['day'].values, n_boot=n_boot, boot_seed=2026)
        summ[key] = summarise(te[target].values, pr, boots[key])
    return runs, boots, summ

def results_table(summ, base_rate, label):
    return pd.DataFrame([{'Feature set': label[k], 'PR-AUC [95% CI]': format_ci(s, 'PR-AUC'),
                          'Lift over base rate': round(s['PR-AUC'] / base_rate, 2),
                          'F2 [95% CI]': format_ci(s, 'F2'), 'MCC [95% CI]': format_ci(s, 'MCC')}
                         for k, s in summ.items()])

def uplift_table(boots, comparisons, label, who):
    rows = []
    for a, b in comparisons:
        row = {'Model': who, 'Comparison': f'{label[a]}  minus  {label[b]}'}
        for m in ['PR-AUC', 'F2']:
            d = paired_difference(boots[a], boots[b], m)
            row[f'{m} difference [95% CI]'] = f"{d['mean diff']:+.4f} [{d['CI low']:+.4f}, {d['CI high']:+.4f}]"
            row[f'{m} real?'] = 'yes' if (d['CI low'] > 0 or d['CI high'] < 0) else 'no (CI includes 0)'
        rows.append(row)
    return pd.DataFrame(rows)

def choose_max_features(sets, tr, va, target, grid, seed=0):
    """Pick max_features for each feature set from the VALIDATION part only (one seed, few trees)."""
    chosen, rows = {}, []
    for key, cols in sets.items():
        sc = {}
        for mf in grid:
            m = RandomForestClassifier(n_estimators=N_TREES_SEL, max_features=mf, min_samples_leaf=5,
                                       random_state=seed, n_jobs=-1).fit(tr[cols], tr[target])
            sc[mf] = average_precision_score(va[target], m.predict_proba(va[cols])[:, 1])
        chosen[key] = max(sc, key=sc.get)
        rows.append({'set': key, 'n features': len(cols), **{f'max_features={g}': round(v, 4) for g, v in sc.items()},
                     'chosen': f'max_features={chosen[key]}'})
    return chosen, pd.DataFrame(rows)

rf_by_key = lambda sel: (lambda key: (lambda s: RandomForestClassifier(n_estimators=N_TREES_ABL, max_features=sel[key],
                                                                      min_samples_leaf=5, random_state=s, n_jobs=-1)))
make_rf3 = lambda s: RandomForestClassifier(n_estimators=N_TREES_ABL, max_features=3, min_samples_leaf=5,
                                            random_state=s, n_jobs=-1)
make_lgb = lambda s: LGBMClassifier(n_estimators=N_LGB, learning_rate=0.05, num_leaves=31, min_child_samples=50,
                                    subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=1.0,
                                    random_state=s, n_jobs=-1, verbose=-1)
make_lr = lambda s: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
IEEE_CMP = [('B', 'A'), ('C', 'A'), ('D', 'A'), ('E', 'A'), ('F', 'A'), ('P', 'A'), ('H', 'G')]
base_rate_i = test_i['isFraud'].mean()
print(f'IEEE test base rate (a model with no skill scores a PR-AUC of about this): {base_rate_i:.4f}')

IEEE test base rate (a model with no skill scores a PR-AUC of about this): 0.0344


Sets up the feature sets and the evaluation helpers. The base rate is 0.0344.

In [ ]:
# 4.6a Choose max_features for the Random Forest on the VALIDATION part (the test part is not touched)
mf_i, mf_table_i = choose_max_features(IEEE_SETS, tr_i, val_i, 'isFraud', MF_GRID_IEEE)
mf_table_i.to_csv('results/ieee_rf_max_features_selection.csv', index=False)
mf_table_i

,set,n features,max_features=3,max_features=sqrt,max_features=0.3,max_features=0.6,chosen
0,A,14,0.1507,0.1507,0.1462,0.1439,max_features=sqrt
1,B,16,0.1597,0.1588,0.1588,0.1608,max_features=0.6
2,C,18,0.1478,0.1454,0.1430,0.1408,max_features=3
3,D,20,0.1567,0.1575,0.1577,0.1588,max_features=0.6
4,E,21,0.1699,0.1676,0.1700,0.1729,max_features=0.6
5,F,22,0.1881,0.1836,0.1822,0.1744,max_features=3
6,G,28,0.5336,0.5331,0.5309,0.5190,max_features=3
7,H,35,0.5165,0.5215,0.5327,0.5210,max_features=0.3
8,P,21,0.1262,0.1247,0.1240,0.1269,max_features=0.6


Picks `max_features` for each feature set on validation only. The spread between settings was small, about 0.002 to 0.016 in validation PR-AUC.

In [ ]:
# 4.6b LightGBM on the IEEE feature sets (the standard model for this dataset)
lgb_runs_i, lgb_boot_i, lgb_summ_i = eval_sets('ieee_lgbm', IEEE_SETS, tr_i, val_i, test_i, 'isFraud',
                                               make_lgb, ABL_SEEDS, ABL_CKPT, N_BOOT)
ieee_lgb_table = results_table(lgb_summ_i, base_rate_i, LABEL)
ieee_lgb_table.to_csv('results/ieee_lgbm_feature_sets.csv', index=False)
ieee_lgb_table

ieee_lgbm_A_656abf seed 42: threshold=0.06  val PR-AUC=0.1459
ieee_lgbm_A_656abf seed 43: threshold=0.06  val PR-AUC=0.1454
ieee_lgbm_A_656abf seed 44: threshold=0.05  val PR-AUC=0.1463
ieee_lgbm_B_673655 seed 42: threshold=0.05  val PR-AUC=0.1766
ieee_lgbm_B_673655 seed 43: threshold=0.06  val PR-AUC=0.1794
ieee_lgbm_B_673655 seed 44: threshold=0.06  val PR-AUC=0.1785
ieee_lgbm_C_c18e7e seed 42: threshold=0.06  val PR-AUC=0.1598
ieee_lgbm_C_c18e7e seed 43: threshold=0.06  val PR-AUC=0.1587
ieee_lgbm_C_c18e7e seed 44: threshold=0.06  val PR-AUC=0.1578
ieee_lgbm_D_25bb9c seed 42: threshold=0.06  val PR-AUC=0.1708
ieee_lgbm_D_25bb9c seed 43: threshold=0.06  val PR-AUC=0.1709
ieee_lgbm_D_25bb9c seed 44: threshold=0.05  val PR-AUC=0.1739
ieee_lgbm_E_69fea1 seed 42: threshold=0.06  val PR-AUC=0.1836
ieee_lgbm_E_69fea1 seed 43: threshold=0.06  val PR-AUC=0.1847
ieee_lgbm_E_69fea1 seed 44: threshold=0.05  val PR-AUC=0.1829
ieee_lgbm_F_63abfa seed 42: threshold=0.06  val PR-AUC=0.1960
ieee_lgb

,Feature set,PR-AUC [95% CI],Lift over base rate,F2 [95% CI],MCC [95% CI]
0,A base,"0.1198 [0.1043, 0.1362]",3.48,"0.2846 [0.2644, 0.3051]","0.1698 [0.1546, 0.1860]"
1,B +gap,"0.1056 [0.0862, 0.1355]",3.07,"0.2952 [0.2748, 0.3124]","0.1793 [0.1632, 0.1946]"
2,C +bursts,"0.1018 [0.0836, 0.1290]",2.96,"0.2956 [0.2705, 0.3148]","0.1795 [0.1604, 0.1949]"
3,D +gap+bursts,"0.1068 [0.0874, 0.1369]",3.10,"0.2976 [0.2753, 0.3169]","0.1815 [0.1625, 0.1967]"
4,E +amt vs history,"0.1117 [0.0910, 0.1411]",3.25,"0.3014 [0.2780, 0.3208]","0.1850 [0.1655, 0.2003]"
5,F +account age,"0.1419 [0.1234, 0.1612]",4.12,"0.3150 [0.2932, 0.3324]","0.1973 [0.1809, 0.2121]"
6,G base+C1-C14,"0.4287 [0.3879, 0.4620]",12.46,"0.4469 [0.4202, 0.4674]","0.3596 [0.3353, 0.3798]"
7,H G+temporal,"0.4413 [0.4030, 0.4748]",12.83,"0.4695 [0.4471, 0.4896]","0.3769 [0.3535, 0.3963]"
8,P placebo of E,"0.1200 [0.1059, 0.1346]",3.49,"0.2866 [0.2660, 0.3072]","0.1718 [0.1563, 0.1864]"


With LightGBM we got 0.429 test PR-AUC with IEEE's own columns (G) and 0.441 after adding our time features (H).

In [ ]:
# 4.6c Random Forest with max_features chosen on validation (longest cell of the IEEE part)
rf_runs_i, rf_boot_i, rf_summ_i = eval_sets('ieee_rf_tuned', IEEE_SETS, tr_i, val_i, test_i, 'isFraud',
                                            rf_by_key(mf_i), ABL_SEEDS, ABL_CKPT, N_BOOT, by_key=True)
ieee_rf_table = results_table(rf_summ_i, base_rate_i, LABEL)
ieee_rf_table.to_csv('results/ieee_rf_feature_sets.csv', index=False)
ieee_rf_table

ieee_rf_tuned_A_2ddb89 seed 42: threshold=0.05  val PR-AUC=0.1496
ieee_rf_tuned_A_2ddb89 seed 43: threshold=0.06  val PR-AUC=0.1492
ieee_rf_tuned_A_2ddb89 seed 44: threshold=0.07  val PR-AUC=0.1521
ieee_rf_tuned_B_f1b9e1 seed 42: threshold=0.06  val PR-AUC=0.1595
ieee_rf_tuned_B_f1b9e1 seed 43: threshold=0.06  val PR-AUC=0.1613
ieee_rf_tuned_B_f1b9e1 seed 44: threshold=0.07  val PR-AUC=0.1615
ieee_rf_tuned_C_252283 seed 42: threshold=0.07  val PR-AUC=0.1460
ieee_rf_tuned_C_252283 seed 43: threshold=0.06  val PR-AUC=0.1472
ieee_rf_tuned_C_252283 seed 44: threshold=0.06  val PR-AUC=0.1460
ieee_rf_tuned_D_d224b8 seed 42: threshold=0.07  val PR-AUC=0.1576
ieee_rf_tuned_D_d224b8 seed 43: threshold=0.06  val PR-AUC=0.1577
ieee_rf_tuned_D_d224b8 seed 44: threshold=0.07  val PR-AUC=0.1561
ieee_rf_tuned_E_801bfb seed 42: threshold=0.07  val PR-AUC=0.1716
ieee_rf_tuned_E_801bfb seed 43: threshold=0.07  val PR-AUC=0.1709
ieee_rf_tuned_E_801bfb seed 44: threshold=0.07  val PR-AUC=0.1701
ieee_rf_tu

,Feature set,PR-AUC [95% CI],Lift over base rate,F2 [95% CI],MCC [95% CI]
0,A base,"0.1054 [0.0950, 0.1173]",3.06,"0.2528 [0.2369, 0.2712]","0.1458 [0.1338, 0.1605]"
1,B +gap,"0.0935 [0.0782, 0.1127]",2.72,"0.2549 [0.2363, 0.2708]","0.1425 [0.1285, 0.1554]"
2,C +bursts,"0.0880 [0.0730, 0.1068]",2.56,"0.2732 [0.2541, 0.2917]","0.1587 [0.1444, 0.1740]"
3,D +gap+bursts,"0.0925 [0.0779, 0.1126]",2.69,"0.2642 [0.2438, 0.2796]","0.1510 [0.1353, 0.1640]"
4,E +amt vs history,"0.0945 [0.0783, 0.1174]",2.75,"0.2749 [0.2537, 0.2926]","0.1611 [0.1435, 0.1743]"
5,F +account age,"0.1067 [0.0873, 0.1347]",3.10,"0.3037 [0.2833, 0.3241]","0.1870 [0.1709, 0.2013]"
6,G base+C1-C14,"0.4167 [0.3816, 0.4508]",12.11,"0.4235 [0.3919, 0.4537]","0.3162 [0.2835, 0.3533]"
7,H G+temporal,"0.4180 [0.3770, 0.4628]",12.15,"0.4408 [0.4076, 0.4751]","0.3417 [0.3018, 0.3907]"
8,P placebo of E,"0.0968 [0.0865, 0.1073]",2.81,"0.2326 [0.2173, 0.2486]","0.1201 [0.1088, 0.1324]"


For the tuned Random Forest, G (IEEE's own columns) scored 0.417 and H (G plus our time features) scored 0.418, so our features add almost nothing to PR-AUC for this model.

In [ ]:
# 4.6d Sensitivity check: Random Forest with the Sparkov setting max_features=3
rf3_runs_i, rf3_boot_i, rf3_summ_i = eval_sets('ieee_rf3', IEEE_SETS, tr_i, val_i, test_i, 'isFraud',
                                               make_rf3, ABL_SEEDS, ABL_CKPT, N_BOOT)
ieee_rf3_table = results_table(rf3_summ_i, base_rate_i, LABEL)
ieee_rf3_table.to_csv('results/ieee_rf_maxfeat3_feature_sets.csv', index=False)
ieee_rf3_table

ieee_rf3_A_35326b seed 42: threshold=0.05  val PR-AUC=0.1496
ieee_rf3_A_35326b seed 43: threshold=0.06  val PR-AUC=0.1492
ieee_rf3_A_35326b seed 44: threshold=0.07  val PR-AUC=0.1521
ieee_rf3_B_06866f seed 42: threshold=0.06  val PR-AUC=0.1591
ieee_rf3_B_06866f seed 43: threshold=0.06  val PR-AUC=0.1611
ieee_rf3_B_06866f seed 44: threshold=0.06  val PR-AUC=0.1611
ieee_rf3_C_252283 seed 42: threshold=0.07  val PR-AUC=0.1460
ieee_rf3_C_252283 seed 43: threshold=0.06  val PR-AUC=0.1472
ieee_rf3_C_252283 seed 44: threshold=0.06  val PR-AUC=0.1460
ieee_rf3_D_6d5213 seed 42: threshold=0.07  val PR-AUC=0.1574
ieee_rf3_D_6d5213 seed 43: threshold=0.07  val PR-AUC=0.1602
ieee_rf3_D_6d5213 seed 44: threshold=0.06  val PR-AUC=0.1583
ieee_rf3_E_55bea2 seed 42: threshold=0.06  val PR-AUC=0.1691
ieee_rf3_E_55bea2 seed 43: threshold=0.07  val PR-AUC=0.1679
ieee_rf3_E_55bea2 seed 44: threshold=0.06  val PR-AUC=0.1688
ieee_rf3_F_0e398c seed 42: threshold=0.07  val PR-AUC=0.1868
ieee_rf3_F_0e398c seed 4

,Feature set,PR-AUC [95% CI],Lift over base rate,F2 [95% CI],MCC [95% CI]
0,A base,"0.1054 [0.0950, 0.1173]",3.06,"0.2528 [0.2369, 0.2712]","0.1458 [0.1338, 0.1605]"
1,B +gap,"0.0898 [0.0736, 0.1122]",2.61,"0.2648 [0.2471, 0.2806]","0.1508 [0.1358, 0.1622]"
2,C +bursts,"0.0880 [0.0730, 0.1068]",2.56,"0.2732 [0.2541, 0.2917]","0.1587 [0.1444, 0.1740]"
3,D +gap+bursts,"0.0950 [0.0789, 0.1189]",2.76,"0.2830 [0.2603, 0.3033]","0.1682 [0.1519, 0.1829]"
4,E +amt vs history,"0.0964 [0.0791, 0.1225]",2.80,"0.2864 [0.2670, 0.3073]","0.1711 [0.1557, 0.1888]"
5,F +account age,"0.1067 [0.0873, 0.1347]",3.10,"0.3037 [0.2833, 0.3241]","0.1870 [0.1709, 0.2013]"
6,G base+C1-C14,"0.4167 [0.3816, 0.4508]",12.11,"0.4235 [0.3919, 0.4537]","0.3162 [0.2835, 0.3533]"
7,H G+temporal,"0.3785 [0.3366, 0.4349]",11.00,"0.4278 [0.3956, 0.4609]","0.3267 [0.2896, 0.3726]"
8,P placebo of E,"0.0897 [0.0783, 0.1016]",2.61,"0.2479 [0.2307, 0.2641]","0.1353 [0.1225, 0.1482]"


With the Sparkov setting `max_features=3` we got 0.417 (G) and 0.379 (H). The drop comes from the setting, not from the data.

In [ ]:
# 4.7 Logistic Regression on the same sets (a linear model cannot use gaps/bursts in a non-linear way, so this is a reference)
lr_runs_i, lr_boot_i, lr_summ_i = eval_sets('ieee_lr', IEEE_SETS, tr_i, val_i, test_i, 'isFraud',
                                            make_lr, [42], ABL_CKPT, N_BOOT)
ieee_lr_table = results_table(lr_summ_i, base_rate_i, LABEL)
ieee_lr_table.to_csv('results/ieee_lr_feature_sets.csv', index=False)
ieee_lr_table

ieee_lr_A_c3a9d8 seed 42: threshold=0.06  val PR-AUC=0.1362
ieee_lr_B_8f6c88 seed 42: threshold=0.06  val PR-AUC=0.1383
ieee_lr_C_4221f8 seed 42: threshold=0.07  val PR-AUC=0.1444
ieee_lr_D_f45505 seed 42: threshold=0.06  val PR-AUC=0.1378
ieee_lr_E_471be4 seed 42: threshold=0.06  val PR-AUC=0.1377
ieee_lr_F_d38ec0 seed 42: threshold=0.07  val PR-AUC=0.1476
ieee_lr_G_5c50e0 seed 42: threshold=0.10  val PR-AUC=0.3973
ieee_lr_H_c11163 seed 42: threshold=0.09  val PR-AUC=0.3884
ieee_lr_P_38291c seed 42: threshold=0.06  val PR-AUC=0.1356


,Feature set,PR-AUC [95% CI],Lift over base rate,F2 [95% CI],MCC [95% CI]
0,A base,"0.1127 [0.1013, 0.1263]",3.27,"0.2903 [0.2665, 0.3134]","0.1745 [0.1546, 0.1929]"
1,B +gap,"0.0915 [0.0784, 0.1106]",2.66,"0.2875 [0.2652, 0.3084]","0.1721 [0.1551, 0.1893]"
2,C +bursts,"0.0881 [0.0744, 0.1064]",2.56,"0.2781 [0.2552, 0.3004]","0.1648 [0.1468, 0.1833]"
3,D +gap+bursts,"0.0991 [0.0869, 0.1155]",2.88,"0.2835 [0.2598, 0.3045]","0.1682 [0.1502, 0.1851]"
4,E +amt vs history,"0.1006 [0.0884, 0.1165]",2.92,"0.2864 [0.2638, 0.3071]","0.1709 [0.1533, 0.1879]"
5,F +account age,"0.1030 [0.0889, 0.1206]",2.99,"0.2906 [0.2678, 0.3136]","0.1755 [0.1578, 0.1946]"
6,G base+C1-C14,"0.3289 [0.2872, 0.3725]",9.56,"0.4005 [0.3718, 0.4254]","0.3129 [0.2869, 0.3376]"
7,H G+temporal,"0.3074 [0.2679, 0.3494]",8.93,"0.3799 [0.3528, 0.4034]","0.2720 [0.2495, 0.2928]"
8,P placebo of E,"0.1127 [0.1011, 0.1264]",3.27,"0.2887 [0.2638, 0.3125]","0.1731 [0.1532, 0.1912]"


With Logistic Regression we got 0.329 (G) and 0.307 (H), so a linear model gets no gain.

In [ ]:
# 4.8 Paired differences (this is the table that answers "does it transfer?")
ieee_uplift = pd.concat([uplift_table(lgb_boot_i, IEEE_CMP, LABEL, 'LightGBM'),
                         uplift_table(rf_boot_i, IEEE_CMP, LABEL, 'Random Forest (max_features tuned)'),
                         uplift_table(rf3_boot_i, IEEE_CMP, LABEL, 'Random Forest (max_features=3)'),
                         uplift_table(lr_boot_i, IEEE_CMP, LABEL, 'Logistic Regression')], ignore_index=True)
ieee_uplift.to_csv('results/ieee_uplift.csv', index=False)
ieee_uplift

,Model,Comparison,PR-AUC difference [95% CI],PR-AUC real?,F2 difference [95% CI],F2 real?
0,LightGBM,B +gap minus A base,"-0.0127 [-0.0263, +0.0076]",no (CI includes 0),"+0.0107 [+0.0026, +0.0191]",yes
1,LightGBM,C +bursts minus A base,"-0.0164 [-0.0293, +0.0043]",no (CI includes 0),"+0.0113 [+0.0019, +0.0199]",yes
2,LightGBM,D +gap+bursts minus A base,"-0.0112 [-0.0262, +0.0120]",no (CI includes 0),"+0.0131 [+0.0040, +0.0216]",yes
3,LightGBM,E +amt vs history minus A base,"-0.0064 [-0.0217, +0.0153]",no (CI includes 0),"+0.0169 [+0.0064, +0.0281]",yes
4,LightGBM,F +account age minus A base,"+0.0221 [+0.0121, +0.0320]",yes,"+0.0305 [+0.0203, +0.0404]",yes
5,LightGBM,P placebo of E minus A base,"-0.0000 [-0.0036, +0.0049]",no (CI includes 0),"+0.0018 [-0.0020, +0.0049]",no (CI includes 0)
6,LightGBM,H G+temporal minus G base+C1-C14,"+0.0128 [+0.0070, +0.0179]",yes,"+0.0232 [+0.0161, +0.0310]",yes
7,Random Forest (max_features tuned),B +gap minus A base,"-0.0109 [-0.0255, +0.0063]",no (CI includes 0),"+0.0019 [-0.0135, +0.0173]",no (CI includes 0)
8,Random Forest (max_features tuned),C +bursts minus A base,"-0.0162 [-0.0305, +0.0012]",no (CI includes 0),"+0.0206 [+0.0063, +0.0365]",yes
9,Random Forest (max_features tuned),D +gap+bursts minus A base,"-0.0116 [-0.0259, +0.0094]",no (CI includes 0),"+0.0113 [-0.0044, +0.0282]",no (CI includes 0)


With LightGBM, adding our time features to IEEE's own columns made the model better: PR-AUC went up by 0.013 and F2 by 0.023, and both gains are real (the 95% intervals are above 0). With shuffled card IDs the gain was zero (PR-AUC -0.0000, F2 +0.0018, both intervals include 0).

### 4.9 Does the basic-feature result hold across periods? Validation versus test, and week by week

In 4.6b and 4.6c the temporal features (set **E**) look **better than the basic features (set A) on validation** but **not better on test**. We check this two ways: (1) the same comparison on validation and on test, and (2) test PR-AUC for A and E in each week of the test period. Validation values are the mean over the three seeds; test values are the mean of the per-seed PR-AUC.

In [ ]:
# 4.9 Validation versus test (E vs A, and H vs G), then week by week on the test part
from sklearn.metrics import average_precision_score
vt = []
for name, runs, summ in [('LightGBM', lgb_runs_i, lgb_summ_i), ('Random Forest (max_features tuned)', rf_runs_i, rf_summ_i)]:
    for k in ['A', 'E', 'G', 'H']:
        vt.append({'Model': name, 'Feature set': LABEL[k],
                   'Validation PR-AUC': np.mean([r['val_pr_auc'] for r in runs[k]]),
                   'Test PR-AUC': summ[k]['PR-AUC']})
vt = pd.DataFrame(vt)
vt.to_csv('results/ieee_validation_vs_test.csv', index=False)
display(vt.round(4))

y_te = test_i['isFraud'].values
wk_te = ((test_i['ts'] - test_i['ts'].min()).dt.days // 7).values
t0_te = test_i['ts'].min().normalize()
weekly = []
for name, runs in [('LightGBM', lgb_runs_i), ('Random Forest (max_features tuned)', rf_runs_i)]:
    for w in sorted(set(wk_te)):
        m = wk_te == w
        pa = np.mean([average_precision_score(y_te[m], r['test_proba'][m]) for r in runs['A']])
        pe = np.mean([average_precision_score(y_te[m], r['test_proba'][m]) for r in runs['E']])
        weekly.append({'Model': name, 'Test week': int(w), 'Starts': (t0_te + pd.Timedelta(days=7 * int(w))).strftime('%Y-%m-%d'),
                       'Transactions': int(m.sum()), 'Frauds': int(y_te[m].sum()),
                       'A base': pa, 'E +amt vs history': pe, 'E minus A': pe - pa})
weekly = pd.DataFrame(weekly)
weekly.to_csv('results/ieee_weekly_E_vs_A.csv', index=False)
display(weekly.round(3))
for name, g in weekly.groupby('Model'):
    print(f"{name}: E beats A in {int((g['E minus A'] > 0).sum())} of {len(g)} test weeks; "
          f"largest loss {g['E minus A'].min():+.3f} (week {int(g.loc[g['E minus A'].idxmin(), 'Test week'])}), "
          f"largest gain {g['E minus A'].max():+.3f} (week {int(g.loc[g['E minus A'].idxmax(), 'Test week'])})")

**Reading 4.9.** Three things stand out.

1. **On validation the temporal features help the basic features, on test they do not.** For LightGBM, validation PR-AUC is 0.184 with E against 0.146 with A, but test PR-AUC is 0.112 against 0.120. The tuned Random Forest goes the same way (validation about 0.171 against 0.150, test 0.0945 against 0.1054). The paired test difference E minus A is not clearly different from zero (LightGBM -0.0064 [-0.0217, +0.0153]).
2. **The test difference changes sign from week to week**, so it is not a steady loss. The weekly table shows weeks where E is better and weeks where it is worse, and one or two weeks (week 4 for both models) carry most of the loss. Each week has only about 560 to 740 frauds, so a single unusual week can move the whole-test number.
3. **The gain on top of IEEE's own columns (H minus G) is the opposite case.** On validation H and G are almost the same (LightGBM about 0.524 for both), but on test H is clearly better (+0.0128).

**What we can and cannot say.** With one chronological split we cannot tell drift (the effect really changes over time) from a handful of unusual days. We therefore treat the basic-feature comparison (E against A) as **inconclusive on real data**, and the LightGBM result on top of C1-C14 as **supported on this test window**, not as a general law. Repeating the evaluation on several time windows would settle it, and we list this as a limit of the study.

## 5. The same temporal features on Sparkov (like-for-like comparison)

Sparkov already has the exact card (`cc_num`), so here the identifier is exact rather than approximated, and Sparkov has **velocity** (spatial). We test four sets with the same protocol: the five non-temporal features without velocity (**SA**), plus the temporal features (**SB**), plus velocity (**SC**), and both (**SD**). This shows how much of the Sparkov gain comes from geography versus time, and lets the paper say whether the temporal part behaves the same on synthetic and real data.

The same two fixes as on IEEE-CIS apply: the **30-day burn-in** (on Sparkov it only removes the first weeks of 2019, where the card history is cut off) and **`max_features` chosen on validation** for each set (the SB and SD sets have 12 and 13 features, so `max_features=3` would penalise them). LightGBM is run as well.

Runtime note: this is the longest part of the notebook (about 1 million training rows per fit). The saved checkpoints make it safe to resume.

In [ ]:
# 5.1 Sparkov feature sets, burn-in, and the choice of max_features on validation
SP_BASE = ['amt_log', 'category_risk', 'is_night', 'age', 'gender_flag']
SP_T = ['gap_log', 'is_first', 'prior_count_log', 'burst_10m', 'burst_1h', 'burst_24h', 'amt_vs_hist']
SP_SETS = {'SA': SP_BASE, 'SB': SP_BASE + SP_T, 'SC': SP_BASE + ['velocity_log'], 'SD': SP_BASE + SP_T + ['velocity_log']}
SP_LABEL = {'SA': 'SA base (no velocity, no temporal)', 'SB': 'SB +temporal', 'SC': 'SC +velocity', 'SD': 'SD +velocity+temporal'}
sp_te = sparkov_test.copy()

t0_sp = pd.to_datetime(train_sub[TIME]).min()
train_sub_b = train_sub[pd.to_datetime(train_sub[TIME]) >= t0_sp + pd.Timedelta(days=BURN_IN_DAYS)]
print(f"Sparkov training rows: {len(train_sub):,} -> {len(train_sub_b):,} after skipping the first {BURN_IN_DAYS} days. "
      f"Share of first-seen rows: before {train_sub['is_first'].mean():.4f}, after {train_sub_b['is_first'].mean():.4f}, "
      f"test {sp_te['is_first'].mean():.4f}")

mf_sp, mf_table_sp = choose_max_features(SP_SETS, train_sub_b, val, 'is_fraud', MF_GRID_SP)
mf_table_sp.to_csv('results/sparkov_rf_max_features_selection.csv', index=False)
mf_table_sp

Sparkov training rows: 1,036,901 -> 985,603 after skipping the first 30 days. Share of first-seen rows: before 0.0009, after 0.0001, test 0.0000


,set,n features,max_features=3,max_features=sqrt,max_features=0.5,chosen
0,SA,5,0.8586,0.8554,0.8554,max_features=3
1,SB,12,0.8868,0.8868,0.8955,max_features=0.5
2,SC,6,0.8730,0.8633,0.8730,max_features=0.5
3,SD,13,0.8745,0.8745,0.8954,max_features=0.5


Skips the first 30 days of Sparkov and picks `max_features`. Training went from 1,036,901 to 985,603 rows. Three of the four sets (SB, SC and SD) chose 0.5, and only SA kept 3.

In [ ]:
# 5.2 LightGBM and Random Forest (max_features tuned) on the four Sparkov sets
sp_runs_l, sp_boot_l, sp_summ_l = eval_sets('sparkov_lgbm', SP_SETS, train_sub_b, val, sp_te, 'is_fraud',
                                            make_lgb, ABL_SEEDS, ABL_CKPT, N_BOOT)
sp_table_l = results_table(sp_summ_l, sp_te['is_fraud'].mean(), SP_LABEL)
sp_table_l.to_csv('results/sparkov_lgbm_feature_sets.csv', index=False)

sp_runs, sp_boot, sp_summ = eval_sets('sparkov_rf_tuned', SP_SETS, train_sub_b, val, sp_te, 'is_fraud',
                                      rf_by_key(mf_sp), ABL_SEEDS, ABL_CKPT, N_BOOT, by_key=True)
sp_table = results_table(sp_summ, sp_te['is_fraud'].mean(), SP_LABEL)
sp_table.to_csv('results/sparkov_feature_sets.csv', index=False)

SP_CMP = [('SB', 'SA'), ('SC', 'SA'), ('SD', 'SC'), ('SB', 'SC'), ('SD', 'SB')]   # SB-SC: time instead of velocity; SD-SB: velocity on top of time
sp_uplift = pd.concat([uplift_table(sp_boot, SP_CMP, SP_LABEL, 'Random Forest (max_features tuned)'),
                       uplift_table(sp_boot_l, SP_CMP, SP_LABEL, 'LightGBM')], ignore_index=True)
sp_uplift.to_csv('results/sparkov_uplift.csv', index=False)
display(sp_table_l); display(sp_table); display(sp_uplift)

Tests the same question with the exact card ID. Time features added +0.0349 PR-AUC for the Random Forest and +0.0645 for LightGBM. On top of velocity they still added +0.0182 and +0.0301.

### 5.3 Findings: temporal or spatial (velocity) on Sparkov?

*(All numbers come from the tables in 5.2 and 6.1. The paired differences are on the same resampled test days; "real" means the 95% interval excludes 0.)*

1. **Time features beat velocity in both models.** With the Random Forest, test PR-AUC is 0.8499 with the temporal features (SB) against 0.8315 with velocity (SC); the paired difference SB minus SC is +0.0183 [+0.0094, +0.0264], real. With LightGBM it is 0.8532 against 0.8179 (+0.0352 [+0.0266, +0.0444], real). The F2 difference is real for LightGBM (+0.0227) but not for the Random Forest (+0.0035, interval includes 0).
2. **Velocity adds nothing on top of the time features.** SD (both) minus SB (time only) is -0.0002 [-0.0017, +0.0013] for the Random Forest, so no change. For LightGBM it is slightly **negative**, -0.0051 [-0.0072, -0.0028], so velocity does not help and costs a little. The reverse does not hold: the time features still add a real gain on top of velocity (SD minus SC: +0.0182 for the Random Forest, +0.0301 for LightGBM).
3. **Both ingredients help on their own.** Against the base set SA, the temporal features add +0.0349 (Random Forest) and +0.0645 (LightGBM) PR-AUC, and velocity adds +0.0167 and +0.0292. All four are real.
4. **A likely reason, not tested here:** velocity is distance divided by the time gap, and distance alone was a weak signal in our EDA, so most of what velocity carries may already be in the time gap.

**Limits:** Sparkov is simulated, so the card IDs are exact and the generator may produce bursts that are easier to detect than real ones. There are three seeds, one chronological split, and LightGBM was not tuned.

## 6. Summary for the paper

In [ ]:
# 6.1 One table: does the temporal signal transfer?
import re
def verdict(txt):
    m, lo, hi = map(float, re.findall(r'[+-]\d+\.\d+', txt))
    return 'improves' if lo > 0 else ('worsens' if hi < 0 else 'no clear change')

def pick(df, who, a, b, label):
    r = df[(df['Model'] == who) & (df['Comparison'] == f'{label[a]}  minus  {label[b]}')].iloc[0]
    return r['PR-AUC difference [95% CI]'], r['F2 difference [95% CI]']

rows = []
for who in ['LightGBM', 'Random Forest (max_features tuned)', 'Random Forest (max_features=3)', 'Logistic Regression']:
    for a, b, what in [('E', 'A', 'temporal features added to the basic features'),
                       ('F', 'A', 'temporal features + account age added to the basic features'),
                       ('H', 'G', 'temporal features added on top of IEEE C1-C14'),
                       ('P', 'A', 'PLACEBO (shuffled identifiers)')]:
        p, f = pick(ieee_uplift, who, a, b, LABEL)
        rows.append({'Dataset': 'IEEE-CIS (real, card id approximated)', 'Model': who, 'What was added': what,
                     'PR-AUC change [95% CI]': p, 'PR-AUC': verdict(p), 'F2 change [95% CI]': f, 'F2': verdict(f)})
for who in ['Random Forest (max_features tuned)', 'LightGBM']:
    for a, b, what in [('SB', 'SA', 'temporal features added (exact card id)'),
                       ('SC', 'SA', 'velocity added (exact card id, has geography)'),
                       ('SD', 'SC', 'temporal features added on top of velocity'),
                       ('SB', 'SC', 'temporal features INSTEAD of velocity (SB vs SC)'),
                       ('SD', 'SB', 'velocity added on top of temporal features (SD vs SB)')]:
        p, f = pick(sp_uplift, who, a, b, SP_LABEL)
        rows.append({'Dataset': 'Sparkov (synthetic, exact card id)', 'Model': who, 'What was added': what,
                     'PR-AUC change [95% CI]': p, 'PR-AUC': verdict(p), 'F2 change [95% CI]': f, 'F2': verdict(f)})
transfer = pd.DataFrame(rows)
transfer.to_csv('results/transfer_summary.csv', index=False)
transfer

One table of everything: IEEE-CIS against Sparkov, every model, with a verdict for each. The IEEE LightGBM rows say "improves" for the gain on top of IEEE's columns, and the placebo says "no clear change".

### 6.2 Findings: does the temporal signal transfer to real data?

*(Every number below is copied from the outputs of Section 4 (tables 4.6 to 4.9) and from table 6.1. "Real" means the 95% paired-bootstrap interval excludes 0.)*

**IEEE-CIS (real data, card identifier approximated)**

1. **It transfers modestly, and the answer depends on the model.**
   * **LightGBM** (the standard model for this dataset): on top of IEEE's own `C1`-`C14` columns, the temporal features add PR-AUC **+0.0128** [+0.0070, +0.0179] and F2 **+0.0232** [+0.0161, +0.0310]; both intervals are above 0. Adding the temporal features **and account age** to the basic features gives PR-AUC +0.0221 [+0.0121, +0.0320] and F2 +0.0305. The **placebo** (shuffled identifiers) gives no gain: PR-AUC -0.0000 [-0.0036, +0.0049], F2 +0.0018 [-0.0020, +0.0049].
   * **Random Forest** (`max_features` chosen on validation): **no change in PR-AUC** on top of `C1`-`C14` (+0.0018 [-0.0069, +0.0118]), but a real **F2** gain (+0.0177 [+0.0090, +0.0258]). On the basic features there is also an F2 gain (E minus A +0.0219, F minus A +0.0509) with no clear PR-AUC change. The **placebo does not reproduce these gains; it makes the model worse**: PR-AUC -0.0088 [-0.0139, -0.0032] and F2 -0.0202 [-0.0318, -0.0086], both real. So the Random Forest F2 gains are not simply the effect of adding extra columns.
   * **Logistic Regression** (linear reference): no gain, and several comparisons are worse (for example H minus G: PR-AUC -0.0215, F2 -0.0209). A linear model cannot use gaps and counts the way the tree models do.
2. **An earlier apparent drop was caused by a model setting, not by the data.** With the Sparkov setting `max_features=3`, adding the temporal features on top of `C1`-`C14` lowered the Random Forest's PR-AUC by 0.0364 [-0.0576, -0.0109]. With `max_features` chosen on validation the change is +0.0018 (no clear change). The paper should not say that the temporal features "hurt" on real data.
3. **The start-of-data cut-off was handled.** Cards look "new" far more often in the first weeks (66% of transactions in week 0 against about 30% later; table in 4.3). All comparisons use training rows after the first 30 days, which brings the share of first-seen rows in training to 0.342, close to validation (0.303) and test (0.299).
4. **Size of the effect.** The vendor columns `C1`-`C14` matter far more: test PR-AUC is 0.4287 (LightGBM) and 0.4167 (Random Forest) with them, against 0.1198 and 0.1054 for the basic set. The temporal features are a small addition on top, not a replacement.
5. **The basic-feature result is not stable across periods (4.9).** For LightGBM the temporal features (E) beat the basic features (A) on validation (0.184 against 0.146) but not on test (0.112 against 0.120), and the tuned Random Forest behaves the same. The test difference changes sign from week to week. On top of `C1`-`C14` the opposite happens: almost no gain on validation (both about 0.524) and +0.0128 on test. The gains therefore depend on the period, and we cannot separate drift from a few unusual days with one split.

**What the paper can claim:** with a card identifier built from `card1` + `addr1` + the `D1` start date, backward-looking time-gap and burst features carry fraud signal on IEEE-CIS (table in 4.3), and for LightGBM they add a small but statistically supported gain on top of the vendor columns that disappears with shuffled identifiers. For the Random Forest the gain is in F2 only. The comparison on the basic features alone is inconclusive.

**Limits to state:** (1) the card identifier is an approximation (IEEE-CIS has no true card id), so some cards are merged or split; (2) IEEE-CIS has no geography, so velocity could not be tested; (3) the absolute time of day is unreliable in IEEE-CIS, so `is_night` was not used; (4) one chronological split, with results that differ between validation and test (4.9); (5) one fixed LightGBM setting (not tuned); (6) the gains are small compared with the vendor count columns; (7) 200 bootstrap resamples and 3 seeds, fewer than the team's final evaluation.

### 6.3 Overall answer for the paper: spatial or temporal?

* **Sparkov (exact card ID, has geography):** the temporal features beat velocity (PR-AUC 0.8499 against 0.8315 for the Random Forest, 0.8532 against 0.8179 for LightGBM), and velocity adds nothing on top of them (SD minus SB about 0 for the Random Forest, slightly negative for LightGBM). This is why the final model uses the time gap in place of implied velocity (whole-test PR-AUC 0.8444 against 0.834).
* **IEEE-CIS (real, no geography):** only the temporal side can be tested. It transfers for LightGBM on top of the vendor columns (+0.0128 PR-AUC), only in F2 for the Random Forest, and not clearly on the basic features, where the result differs between validation and test.
* **Drift (Sparkov):** see 3.6.
* **What stays open:** a real dataset with geography, so that the spatial side can be tested on real data, and several time windows on IEEE-CIS.

In [ ]:
# [COLAB-ONLY] 6.4 Zip the results so you can download them and commit them to the repo (results/ folder)
import os, zipfile
from google.colab import files
here = os.path.abspath('results')
mine = ['sparkov_monthly_drift.csv', 'sparkov_monthly_psi.csv', 'sparkov_monthly_drift.png', 'transfer_summary.csv',
        'ieee_card_history_by_week.csv', 'ieee_rf_max_features_selection.csv', 'ieee_lgbm_feature_sets.csv',
        'ieee_rf_feature_sets.csv', 'ieee_rf_maxfeat3_feature_sets.csv', 'ieee_lr_feature_sets.csv', 'ieee_uplift.csv',
        'sparkov_rf_max_features_selection.csv', 'sparkov_lgbm_feature_sets.csv', 'sparkov_feature_sets.csv', 'sparkov_uplift.csv',
        'ieee_validation_vs_test.csv', 'ieee_weekly_E_vs_A.csv']
zp = os.path.abspath('transfer_and_drift_results.zip')
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in mine:                                   # only the files made by this notebook (not the team's .joblib files)
        if os.path.exists(os.path.join(here, f)):
            z.write(os.path.join(here, f), arcname='results/' + f)
        else:
            print('missing (did you run every cell?):', f)
print('zip written:', zp, os.path.getsize(zp), 'bytes')
files.download(zp)

zip written: /content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/transfer_and_drift_results.zip 198500 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>